In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=12289, hidden_size_1=256, hidden_size_2=128, hidden_size_3=64,
                 output_size=2):
        super(FeedforwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc3 = nn.Linear(hidden_size_2, hidden_size_3)
        self.fc4 = nn.Linear(hidden_size_3, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, input_data):
        out = self.fc1(input_data)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.tanh(out)
        return out

In [3]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_road_version_4.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_speed_version_4.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_for_lstm_wheel_4.csv')

In [4]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [5]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

13037
13037
13037


In [6]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([ 0.,  0.,  0.,  ...,  0.,  0., 59.], device='cuda:0')
torch.Size([12289])


In [7]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([0.0053, 0.0053], device='cuda:0')
torch.Size([2])


In [8]:
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=128, shuffle=False)

In [9]:
model = FeedforwardNet().cuda()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)

# Обучение модели
num_epochs = 12
for epoch in range(num_epochs):
    for X, y in train_loader:
        # Прямое прохождение
        output = model(X)
        loss = criterion(output, y)

        # Обратное распространение и обновление весов
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

Epoch [1/12], Loss: 0.000176474
Epoch [2/12], Loss: 0.000113146
Epoch [3/12], Loss: 0.000080949
Epoch [4/12], Loss: 0.000052578
Epoch [5/12], Loss: 0.000039623
Epoch [6/12], Loss: 0.000038523
Epoch [7/12], Loss: 0.000051245
Epoch [8/12], Loss: 0.000027466
Epoch [9/12], Loss: 0.000025296
Epoch [10/12], Loss: 0.000022406
Epoch [11/12], Loss: 0.000014252
Epoch [12/12], Loss: 0.000018365


In [10]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_version_3.pth')